# CRISPR Observability
Inspect the frozen bounds and run a selected offline replay. Start from the repository root or its reproducibility directory.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
from datetime import datetime, timezone

root = Path.cwd()
if not (root / 'interactive').is_dir():
    root = root.parent
model = json.loads((root / 'interactive/supergraph/review_graph_model.json').read_bytes())
rows = model['context_metrics']['witness_observations']
assert all(r['E'] + r['NE'] + r['U'] == r['N'] for r in rows)
print(f'{len(rows)} frozen observation partitions verified')


In [ ]:
import altair as alt
import pandas as pd

table = pd.DataFrame(rows)
alt.Chart(table).mark_rule(strokeWidth=5).encode(
    x=alt.X('L_pp:Q', title='Percent of raw records'),
    x2='H_pp:Q',
    y=alt.Y('condition:N', title=None),
    color=alt.Color('event:N', scale=alt.Scale(range=['#527ed7','#9664cd','#e38fae'])),
    tooltip=['preparation:N','run:N','N:Q','U:Q','L_pp:Q','H_pp:Q']
).facet(row='preparation:N').resolve_scale(x='independent')


In [ ]:
section = 'germline'  # all, editing, or germline
mode = 'quick'       # quick, counts, or full
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
destination = Path.home() / 'CRISPR-Observability' / f'{stamp}-{section}-{mode}'
subprocess.run([sys.executable, '-B', str(root / 'reproducibility/reproduce.py'),
                '--section', section, '--mode', mode, '--output', str(destination)],
               cwd=root, check=True)
json.loads((destination / 'REPRODUCTION.json').read_bytes())
